In [ ]:
import os
import gc
import glob
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import KDTree
from sklearn.preprocessing import StandardScaler

from catboost import CatBoostRegressor

DATA_PATH = "/kaggle/input/competitions/rogii-wellbore-geology-prediction"

TRAIN_DIR = f"{DATA_PATH}/train"
TEST_DIR  = f"{DATA_PATH}/test"

TARGET = "TVT"

N_SPLITS = 5
SEED = 42

def seed_everything(seed):

    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(SEED)

train_files = sorted(
    glob.glob(f"{TRAIN_DIR}/*__horizontal_well.csv")
)

test_files = sorted(
    glob.glob(f"{TEST_DIR}/*__horizontal_well.csv")
)

print(len(train_files), len(test_files))

def build_dataset(files):

    dfs = []

    for file in tqdm(files):

        try:

            df = pd.read_csv(file)

            well_id = (
                os.path.basename(file)
                .replace("__horizontal_well.csv", "")
            )

            df["well_id"] = well_id

            dfs.append(df)

        except:
            continue

    return pd.concat(dfs, ignore_index=True)

train = build_dataset(train_files)
test  = build_dataset(test_files)

print(train.shape)
print(test.shape)

BASE_FEATURES = [
    "MD",
    "X",
    "Y",
    "Z",
    "GR",
    "TVT_input"
]

FEATURES = []

for col in BASE_FEATURES:

    if col in train.columns and col in test.columns:
        FEATURES.append(col)

print(FEATURES)

for col in FEATURES:

    med = train[col].median()

    train[col] = train[col].fillna(med)
    test[col]  = test[col].fillna(med)

for col in FEATURES:

    train[col] = train[col].astype(np.float32)
    test[col]  = test[col].astype(np.float32)

train[TARGET] = train[TARGET].astype(np.float32)

train["MD_diff"] = (
    train.groupby("well_id")["MD"]
    .diff()
    .fillna(0)
)

test["MD_diff"] = (
    test.groupby("well_id")["MD"]
    .diff()
    .fillna(0)
)

train["MD_pct"] = (
    train.groupby("well_id")["MD"]
    .rank(pct=True)
)

test["MD_pct"] = (
    test.groupby("well_id")["MD"]
    .rank(pct=True)
)

FEATURES += [
    "MD_diff",
    "MD_pct"
]

for col in FEATURES.copy():

    for lag in [1,2,3]:

        train[f"{col}_lag_{lag}"] = (
            train.groupby("well_id")[col]
            .shift(lag)
        )

        test[f"{col}_lag_{lag}"] = (
            test.groupby("well_id")[col]
            .shift(lag)
        )

        FEATURES.append(f"{col}_lag_{lag}")

for col in FEATURES.copy():

    if "lag" in col:
        continue

    for w in [3,5]:

        train[f"{col}_rollmean_{w}"] = (

            train.groupby("well_id")[col]
            .rolling(w, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)

        )

        test[f"{col}_rollmean_{w}"] = (

            test.groupby("well_id")[col]
            .rolling(w, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)

        )

        FEATURES.append(f"{col}_rollmean_{w}")

train["XY"] = train["X"] * train["Y"]
test["XY"] = test["X"] * test["Y"]

train["XZ"] = train["X"] * train["Z"]
test["XZ"] = test["X"] * test["Z"]

train["YZ"] = train["Y"] * train["Z"]
test["YZ"] = test["Y"] * test["Z"]

train["R"] = np.sqrt(
    train["X"]**2 +
    train["Y"]**2 +
    train["Z"]**2
)

test["R"] = np.sqrt(
    test["X"]**2 +
    test["Y"]**2 +
    test["Z"]**2
)

FEATURES += [
    "XY",
    "XZ",
    "YZ",
    "R"
]

coord_cols = ["X","Y","Z"]

scaler = StandardScaler()

train_coords = scaler.fit_transform(
    train[coord_cols]
)

test_coords = scaler.transform(
    test[coord_cols]
)

tree = KDTree(train_coords)

distances, indices = tree.query(
    test_coords,
    k=20
)

nearest_tvt = []
neighbor_std = []
neighbor_dist = []

for d, inds in zip(distances, indices):

    vals = train.iloc[inds][TARGET].values

    nearest_tvt.append(vals.mean())
    neighbor_std.append(vals.std())
    neighbor_dist.append(d.mean())

test["nearest_tvt"] = nearest_tvt
test["neighbor_std"] = neighbor_std
test["neighbor_dist"] = neighbor_dist

train_tree = KDTree(train_coords)

d_train, i_train = train_tree.query(
    train_coords,
    k=20
)

train_nearest = []
train_std = []
train_dist = []

for d, inds in zip(d_train, i_train):

    vals = train.iloc[inds][TARGET].values

    train_nearest.append(vals.mean())
    train_std.append(vals.std())
    train_dist.append(d.mean())

train["nearest_tvt"] = train_nearest
train["neighbor_std"] = train_std
train["neighbor_dist"] = train_dist

FEATURES += [
    "nearest_tvt",
    "neighbor_std",
    "neighbor_dist"
]

FEATURES = list(sorted(set(FEATURES)))

for col in FEATURES:

    med = train[col].median()

    train[col] = train[col].fillna(med)
    test[col]  = test[col].fillna(med)

train["residual"] = (
    train[TARGET] -
    train["nearest_tvt"]
)

gkf = GroupKFold(n_splits=N_SPLITS)

oof = np.zeros(len(train))
preds = np.zeros(len(test))

for fold, (tr_idx, va_idx) in enumerate(

    gkf.split(
        train,
        train["residual"],
        groups=train["well_id"]
    )

):

    print("FOLD:", fold)

    X_train = train.iloc[tr_idx][FEATURES]
    y_train = train.iloc[tr_idx]["residual"]

    X_valid = train.iloc[va_idx][FEATURES]
    y_valid = train.iloc[va_idx]["residual"]

    X_test = test[FEATURES]

    model = CatBoostRegressor(

        iterations=3500,
        learning_rate=0.02,
        depth=10,

        loss_function="RMSE",
        eval_metric="RMSE",

        task_type="GPU",

        random_seed=SEED,

        verbose=500
    )

    model.fit(

        X_train,
        y_train,

        eval_set=(X_valid, y_valid),

        early_stopping_rounds=300,

        use_best_model=True
    )

    valid_pred = model.predict(X_valid)

    test_pred = model.predict(X_test)

    oof[va_idx] = (
        train.iloc[va_idx]["nearest_tvt"].values +
        valid_pred
    )

    preds += (
        test["nearest_tvt"].values +
        test_pred
    ) / N_SPLITS

    del X_train
    del X_valid
    del y_train
    del y_valid

    gc.collect()

rmse = np.sqrt(
    mean_squared_error(
        train[TARGET],
        oof
    )
)

print("CV:", rmse)

test["preds"] = preds

test["preds"] = (
    test.groupby("well_id")["preds"]
    .transform(
        lambda x: x.rolling(
            11,
            center=True,
            min_periods=1
        ).mean()
    )
)

preds = test["preds"].values

preds = np.nan_to_num(preds)

sample_sub = pd.read_csv(
    f"{DATA_PATH}/sample_submission.csv"
)

target_col = sample_sub.columns[1]

preds = preds[:len(sample_sub)]

sample_sub[target_col] = preds

sample_sub.to_csv(
    "submission.csv",
    index=False
)

print(sample_sub.head())

fi = pd.DataFrame({

    "feature": FEATURES,
    "importance": model.feature_importances_

})

fi = fi.sort_values(
    by="importance",
    ascending=False
)

print(fi.head(30))

fi.to_csv(
    "feature_importance.csv",
    index=False
)

print("DONE")